# Physical AI Lab — Module 0: Kickoff

## The Physical AI Pipeline

Every system in this lab follows the same closed loop:

```
[Webcam] → [Perception Layer] → [State Vector] → [Sim Agent] → [Action] → [World/Sim] → (back to Webcam)
```

The loop is **closed**: every action the agent takes changes the environment, which changes what the camera sees on the next frame.  
A robot arm that can't sense its own position can't correct its errors. A camera that can't feed back into the controller is just a recorder, not a sensor.

---

### Stage-by-stage breakdown

| Stage | What it does | Code you'll write |
|---|---|---|
| **Webcam** | Captures raw pixel data — the robot's eye. | `modules/01_perception/01_webcam_basics.py` |
| **Perception Layer** | Extracts structured information from pixels — e.g. hand landmark positions. | `modules/01_perception/02_hand_tracking.py` |
| **State Vector** | A compact `float32` array encoding everything the agent needs. | `modules/01_perception/03_joint_angles.py` |
| **Sim Agent** | A policy (rule or neural network) that maps state → action. | `modules/03_rl/02_train_cartpole.py` |
| **Action** | Numeric commands sent to the sim or robot — e.g. joint torques. | `modules/04_perception_to_action/01_hand_to_reacher.py` |
| **World/Sim** | The environment (MuJoCo or physical robot) that executes the action. | `modules/02_simulation/02_mujoco_reacher.py` |

---

### Why the loop matters

Static inference tasks (e.g. image classification) run the pipeline **once** and stop.  
Physical AI runs it **continuously** — each action feeds back into the next observation.  
This is what makes the system *physical*: the robot's behaviour shapes its own future inputs.

---

### What you will build across the six modules

| Module | Pipeline stage | Key file |
|---|---|---|
| 0 — Kickoff | Full overview (here) | `modules/00_kickoff/overview.ipynb` |
| 1 — Perception | Webcam → State Vector | `modules/01_perception/exercise.py` |
| 2 — Simulation | State Vector → World/Sim | `modules/02_simulation/exercise.py` |
| 3 — RL | Training a Sim Agent | `modules/03_rl/exercise.py` |
| 4 — Perception to Action | Closed loop | `modules/04_perception_to_action/exercise.py` |
| 5 — Foundation Models | VLM reasoning layer | `modules/05_foundation_models/exercise.py` |

## The State Vector

The **state vector** is the central data structure of the Physical AI pipeline.  
It is a 1D NumPy array of `float32` values — compact enough for a neural network to process at 50–100 Hz, yet rich enough to capture everything the agent needs to act.

Run the cell below to create a sample state vector and inspect its properties.

In [ ]:
import numpy as np

# ── What is a state vector? ───────────────────────────────────────────────
# A state vector encodes the observable state of the world at one timestep
# as a 1D array of numbers.  The numbers can mean anything — joint angles,
# velocities, positions, sensor readings — as long as they are consistently
# defined and the agent always receives them in the same order.
#
# This example represents Module 1's hand joint angles (in degrees):
#   state[0] = theta_thumb   (opening angle of thumb at MCP joint)
#   state[1] = theta_index   (opening angle of index finger at MCP joint)
#   state[2] = theta_middle  (opening angle of middle finger at MCP joint)
#   state[3] = theta_ring    (opening angle of ring finger — you'll add this!)
state = np.array([145.2, 163.8, 171.4, 168.9], dtype=np.float32)

print("Sample hand state vector (Module 1):")
print(f"  {state}")
print()
print(f"Shape:  {state.shape}   ← (4,) means a 1D array with 4 elements")
print(f"Dtype:  {state.dtype}  ← float32 uses half the memory of float64")
print()
print("Interpretation:")
print(f"  θ_thumb  = {state[0]:.1f}°")
print(f"  θ_index  = {state[1]:.1f}°")
print(f"  θ_middle = {state[2]:.1f}°")
print(f"  θ_ring   = {state[3]:.1f}°")
print()
print("0° = fully closed (fist), ~180° = fully open (flat hand)")

## State Vectors Across the Pipeline

Each module uses a different state vector.  The shape varies, but the format is always the same — a flat `float32` array.

| Module | State vector contents | Shape |
|---|---|---|
| 1 — Perception | [θ_thumb, θ_index, θ_middle, θ_ring] in degrees | `(4,)` |
| 2 — CartPole | [cart_pos, cart_vel, pole_angle, pole_ang_vel] | `(4,)` |
| 2 — Reacher | cos/sin joint angles, target xy, velocities, fingertip-target vector | `(10,)` |
| 4 — Hand→Reacher | Two joint torques derived from finger position | `(2,)` |

Run the cell below to see how different state vector shapes look side by side.

In [ ]:
import numpy as np

# ── Compare state vectors across modules ──────────────────────────────────
# All different shapes, all float32, all 1D arrays.

# Module 1: hand joint angles from MediaPipe landmarks
hand_state = np.array([145.2, 163.8, 171.4, 168.9], dtype=np.float32)

# Module 2: CartPole observation (from Gymnasium env.reset())
cartpole_state = np.array([-0.02, 0.01, 0.03, -0.01], dtype=np.float32)
#                           cart_pos  vel  pole_angle  ang_vel

# Module 4: action vector sent to Reacher-v5 (torques, not positions)
reacher_action = np.array([0.45, -0.32], dtype=np.float32)
#                           joint1_torque  joint2_torque

print("State / action vectors across modules:")
print()
print(f"Module 1 hand angles:   {hand_state}  shape={hand_state.shape}")
print(f"Module 2 CartPole obs:  {cartpole_state}  shape={cartpole_state.shape}")
print(f"Module 4 Reacher torque: {reacher_action}  shape={reacher_action.shape}")
print()
print("All are float32 numpy arrays — same data type, different shapes.")
print("The pipeline stages are connected by passing these arrays between functions.")

## Next steps

You've seen the concept — now see it in code:

1. Open `modules/01_perception/01_webcam_basics.py` — the first stage: capturing frames
2. Open `modules/01_perception/03_joint_angles.py` — computing the state vector from landmarks  
3. Run `modules/01_perception/exercise.py` — your first coding task: add the ring finger angle

The hub page explains each step interactively.  Open it in your browser at **http://localhost:8000**.